<a href="https://colab.research.google.com/github/fourmansyah/Mapperatorinator/blob/main/colab/mapperatorinator_inference_with_preset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beatmap Generation with Mapperatorinator [MOD]

This notebook is an better interactive demo of an osu! beatmap generation model created by OliBomby [MOD]. This model is capable of generating hit objects, hitsounds, timing, kiai times, and SVs for all 4 gamemodes and can do batch osu file. You can upload a beatmap to give to the model as additional context or remap parts of the beatmap.

### Instructions for running:

* Read and accept the rules regarding using this tool by clicking the checkbox.
* Make sure to use a GPU runtime, click:  __Runtime >> Change Runtime Type >> GPU__
* __Execute each cell in order__. Press ▶️ on the left of each cell to execute the cell.
* __Setup Environment__: run the first cell to clone the repository and install the required dependencies. You only need to run this cell once per session.
* __Upload Audio__: choose a .mp3 or .ogg file from your computer.
* __Upload Beatmap__: optionally choose a beatmap .osu file from your computer.  You can find these files in stable by using File > Open Song Folder, or in lazer by using File > Edit Externally.
* __Configure__: choose your generation parameters to control the style of the generated beatmap.
* Generate the beatmap using the __Generate Beatmap__ cell. (it may take a few minutes depending on the length of the song)


In [ ]:
#@title 🚀 Setup Environment { display-mode: "form" }
#@markdown ### Use this tool responsibly. Always disclose the use of AI in your beatmaps. Accept the rules and run this cell.
i_accept_the_rules = False # @param {type:"boolean"}

assert i_accept_the_rules, "Read and accept the rules first!"

!git clone https://github.com/fourmansyah/Mapperatorinator.git
%cd Mapperatorinator

!pip install transformers==4.57.3
!pip install hydra-core nnaudio
!pip install slider git+https://github.com/llllllllll/slider.git
!pip install rosu-pp-py git+https://github.com/MaxOhn/rosu-pp-py.git
!pip install yt-dlp
from google.colab import files

import os
from hydra import compose, initialize_config_dir
from osuT5.osuT5.event import ContextType
from inference import main

output_path = "output"
input_audio = ""
input_beatmap = ""

In [ ]:
#@title Local Upload Audio { display-mode: "form" }
#@markdown Run this cell to upload audio. This is the song to generate a beatmap for. Please upload a .mp3 or .ogg file.

def upload_audio():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Multiple files uploaded; using only one.')
    file = data[0]
    if not file.endswith('.mp3') and not file.endswith('.ogg'):
        print('Invalid file format. Please upload a .mp3 or .ogg file.')
        return ""
    return data[0]

input_audio = upload_audio()

In [ ]:
#@title 🎵 Download Audio via URL { display-mode: "form" }
#@markdown Enter the YouTube URL or direct link to the audio file. The system will automatically download and display the music player.
audio_url = "" #@param {type:"string"}

import os
import re
import subprocess
from IPython.display import Audio, display

def download_and_validate_audio(url):
    """Download audio from URL with proper error handling and audio player"""
    global audio_path

    if not url or url.strip() == "":
        print("❌ URL kosong! Harap masukkan link yang valid.")
        return None

    print(f"⏳ Mendownload audio dari: {url} ...")

    # Kita buat nama file bersih agar tidak ada karakter aneh dari judul YouTube
    clean_filename = "audio"

    # Hapus file lama agar tidak tertumpuk
    for ext in ['.mp3', '.ogg', '.wav']:
        if os.path.exists(f"{clean_filename}{ext}"):
            os.remove(f"{clean_filename}{ext}")

    try:
        # Perintah yt-dlp untuk ekstrak audio MP3
        command = [
            "yt-dlp",
            "-x",
            "--audio-format", "mp3",
            "--audio-quality", "0",
            "-o", f"{clean_filename}.%(ext)s",
            url
        ]

        # Menjalankan perintah secara diam-diam agar tampilan Colab tetap rapi
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

        # Path file hasil download
        audio_path = os.path.abspath(f"{clean_filename}.mp3")

        if os.path.exists(audio_path):
            print(f"✅ Audio berhasil didownload!")
            print(f"Saved as: {clean_filename}.mp3")
            print(f"Path: {audio_path}")

            # Memunculkan audio player seperti di kode dasar milikmu
            display(Audio(audio_path))

            return audio_path
        else:
            print("❌ File gagal ditemukan setelah proses download selesai.")
            return None

    except subprocess.CalledProcessError:
        print("❌ Terjadi kesalahan saat mendownload. Pastikan URL valid dan video tidak diprivat.")
        return None
    except Exception as e:
        print(f"❌ Error sistem: {e}")
        return None

# Eksekusi fungsi download
audio_path = download_and_validate_audio(audio_url)

# Meneruskan variabel ke input_audio agar Segmen 3 (AI Generator) bisa membacanya
input_audio = audio_path

if not audio_path:
    print("\n⚠️ Silakan periksa kembali URL kamu dan jalankan ulang cell ini.")

In [ ]:
#@title (Optional) Upload Beatmap { display-mode: "form" }
#@markdown This step is required if you want to use `in_context` or `add_to_beatmap` to provide additional info to the model.
#@markdown It will also fill in any missing metadata and unknown values in the configuration using info of the reference beatmap.
#@markdown Please upload a **.osu** file. You can find the .osu file in the song folder in stable or by using File > Edit Externally in lazer.
use_reference_beatmap = False # @param {type:"boolean"}

def upload_beatmap():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Multiple files uploaded; using only one.')
    file = data[0]
    if not file.endswith('.osu'):
        print('Invalid file format. Please upload a .osu file.\nIn stable you can find the .osu file in the song folder (File > Open Song Folder).\nIn lazer you can find the .osu file by using File > Edit Externally.')
        return ""
    return file

if use_reference_beatmap:
    input_beatmap = upload_beatmap()
else:
    input_beatmap = ""

In [ ]:
#@title 🎚️ If you don't understand how to set it, Just Use this Preset [Generator v31]

# =========================
# PILIH PRESET
# =========================
model = "Mapperatorinator V31"  # @param ["Mapperatorinator V29", "Mapperatorinator V30", "Mapperatorinator V31"]
gamemode = "standard"  # @param ["standard", "taiko", "catch the beat", "mania"]
diff_preset = "Normal"  # @param ["Normal", "Hard", "Insane", "Expert", "Extra"]

# =========================
# METADATA SAJA
# =========================
title = ""  # @param {type:"string"}
artist = ""  # @param {type:"string"}
creator = "Ai"  # @param {type:"string"}
version_name = ""  # @param {type:"string"}
tags = ""  # @param {type:"string"}
background = ""  # @param {type:"string"}
preview_time = -1  # @param {type:"integer"}

# =========================
# OPSIONAL
# =========================
mapper_id = -1  # @param {type:"integer"}
beatmap_id = -1  # @param {type:"integer"}
year = 2023  # @param {type:"integer"}
lora_path = ""  # @param ["fourmansyah/LoRA-2025", "fourmansyah/LoRA-sliderSlop", "fourmansyah/LoRA-aim-control", "fourmansyah/LoRA-Arles-v1", "fourmansyah/LoRA-Kroytz"] {allow-input: true}

hitsounded = True  # @param {type:"boolean"}
generate_positions = False  # @param {type:"boolean"}

start_time = -1  # @param {type:"integer"}
end_time = -1  # @param {type:"integer"}
in_context = "[NONE]"  # @param ["[NONE]", "[TIMING]", "[TIMING,KIAI]", "[TIMING,KIAI,MAP]", "[GD,TIMING,KIAI]", "[NO_HS,TIMING,KIAI]"]
output_type = "[MAP]"  # @param ["[MAP]", "[TIMING,KIAI,MAP,SV]", "[TIMING]"]
add_to_beatmap = False  # @param {type:"boolean"}

seed = -1  # @param {type:"integer"}
quantity = 1  # @param {type:"slider", min:1, max:10, step:1}
download_mode = "Individual .osu Files"  # @param ["Master .osz (Lagu + Semua Map)", "Individual .osu Files"]

# =========================
# IMPORT
# =========================
import os
import time
import random
import zipfile
import shutil
from hydra import compose, initialize_config_dir
from osuT5.osuT5.event import ContextType
from inference import main
from google.colab import files

# =========================
# PRESET TERPISAH
# =========================
PRESETS = {
    "Normal": {
        "difficulty": 2.2,
        "hp_drain_rate": 3.5,
        "circle_size": 4,
        "overall_difficulty": 5.5,
        "approach_rate": 6.5,
        "slider_multiplier": 1.6,
        "slider_tick_rate": 1,

        "descriptors": ["clean", "simple", "flow aim"],
        "negative_descriptors": ["chaotic", "visually dense", "tech"],

        "cfg_scale": 1.4,
        "temperature": 0.96,
        "top_p": 0.95,
        "top_k": 0,
        "num_beams": 1,
        "do_sample": True,
        "timeshift_bias": 0.0,

        "timing_leniency": 22,
        "timing_temperature": 0.10,
        "timer_cfg_scale": 1.0,
        "timer_num_beams": 2,
        "timer_iterations": 20,
        "timer_bpm_threshold": 0.1,
        "super_timing": False,
        "bpm": None,
        "offset": None,
        "resnap_events": True,

        "diff_cfg_scale": 1.0,
        "refine_iters": 8,
        "random_init": False,

        "lookback": 0.5,
        "lookahead": 0.4,
    },

    "Hard": {
        "difficulty": 3.6,
        "hp_drain_rate": 4.5,
        "circle_size": 4,
        "overall_difficulty": 6.8,
        "approach_rate": 8.0,
        "slider_multiplier": 1.5,
        "slider_tick_rate": 1,

        "descriptors": ["clean", "flow aim", "bursts"],
        "negative_descriptors": ["chaotic", "visually dense", "overlap reading"],

        "cfg_scale": 1.6,
        "temperature": 0.94,
        "top_p": 0.95,
        "top_k": 0,
        "num_beams": 1,
        "do_sample": True,
        "timeshift_bias": 0.0,

        "timing_leniency": 20,
        "timing_temperature": 0.09,
        "timer_cfg_scale": 1.0,
        "timer_num_beams": 2,
        "timer_iterations": 20,
        "timer_bpm_threshold": 0.1,
        "super_timing": False,
        "bpm": None,
        "offset": None,
        "resnap_events": True,

        "diff_cfg_scale": 1.1,
        "refine_iters": 10,
        "random_init": False,

        "lookback": 0.5,
        "lookahead": 0.4,
    },

    "Insane": {
        "difficulty": 4.8,
        "hp_drain_rate": 5.5,
        "circle_size": 4,
        "overall_difficulty": 8.0,
        "approach_rate": 9.0,
        "slider_multiplier": 1.4,
        "slider_tick_rate": 1,

        "descriptors": ["clean", "jump aim", "flow aim"],
        "negative_descriptors": ["messy", "chaotic", "visually dense"],

        "cfg_scale": 1.8,
        "temperature": 0.92,
        "top_p": 0.95,
        "top_k": 0,
        "num_beams": 1,
        "do_sample": True,
        "timeshift_bias": 0.0,

        "timing_leniency": 18,
        "timing_temperature": 0.08,
        "timer_cfg_scale": 1.0,
        "timer_num_beams": 2,
        "timer_iterations": 20,
        "timer_bpm_threshold": 0.1,
        "super_timing": False,
        "bpm": None,
        "offset": None,
        "resnap_events": True,

        "diff_cfg_scale": 1.2,
        "refine_iters": 12,
        "random_init": False,

        "lookback": 0.5,
        "lookahead": 0.4,
    },

    "Expert": {
        "difficulty": 5.6,
        "hp_drain_rate": 6.0,
        "circle_size": 4,
        "overall_difficulty": 8.7,
        "approach_rate": 9.4,
        "slider_multiplier": 1.4,
        "slider_tick_rate": 1,

        "descriptors": ["jump aim", "precision", "bursts"],
        "negative_descriptors": ["messy", "chaotic", "overlap reading"],

        "cfg_scale": 2.0,
        "temperature": 0.90,
        "top_p": 0.94,
        "top_k": 0,
        "num_beams": 1,
        "do_sample": True,
        "timeshift_bias": 0.0,

        "timing_leniency": 16,
        "timing_temperature": 0.08,
        "timer_cfg_scale": 1.1,
        "timer_num_beams": 2,
        "timer_iterations": 20,
        "timer_bpm_threshold": 0.1,
        "super_timing": False,
        "bpm": None,
        "offset": None,
        "resnap_events": True,

        "diff_cfg_scale": 1.25,
        "refine_iters": 14,
        "random_init": False,

        "lookback": 0.5,
        "lookahead": 0.4,
    },

    "Extra": {
        "difficulty": 6.4,
        "hp_drain_rate": 6.5,
        "circle_size": 4,
        "overall_difficulty": 9.2,
        "approach_rate": 9.7,
        "slider_multiplier": 1.3,
        "slider_tick_rate": 1,

        "descriptors": ["jump aim", "precision", "alt"],
        "negative_descriptors": ["messy", "chaotic", "visually dense"],

        "cfg_scale": 2.2,
        "temperature": 0.88,
        "top_p": 0.93,
        "top_k": 0,
        "num_beams": 1,
        "do_sample": True,
        "timeshift_bias": 0.0,

        "timing_leniency": 15,
        "timing_temperature": 0.07,
        "timer_cfg_scale": 1.1,
        "timer_num_beams": 2,
        "timer_iterations": 20,
        "timer_bpm_threshold": 0.1,
        "super_timing": False,
        "bpm": None,
        "offset": None,
        "resnap_events": True,

        "diff_cfg_scale": 1.3,
        "refine_iters": 16,
        "random_init": False,

        "lookback": 0.55,
        "lookahead": 0.45,
    },
}

preset = PRESETS[diff_preset]

# =========================
# KONVERSI PARAMETER
# =========================
a_config = model.split(" ")[-1].lower()
a_gamemode = ["standard", "taiko", "catch the beat", "mania"].index(gamemode)

a_title = None if title == "" else title
a_artist = None if artist == "" else artist
a_creator = None if creator == "" else creator
a_tags = None if tags == "" else tags
a_background = None if background == "" else background
a_preview_time = None if preview_time == -1 else preview_time
a_mapper_id = None if mapper_id == -1 else mapper_id
a_beatmap_id = None if beatmap_id == -1 else beatmap_id
a_year = None if year == -1 else year
a_lora_path = None if lora_path == "" else lora_path
a_start_time = None if start_time == -1 else start_time
a_end_time = None if end_time == -1 else end_time
a_in_context = [ContextType(c.lower()) for c in in_context[1:-1].split(",")]
a_output_type = [ContextType(c.lower()) for c in output_type[1:-1].split(",")]
a_seed = None if seed == -1 else seed

# pakai nama preset sebagai version kalau kosong
a_base_version = version_name if version_name.strip() else diff_preset

# =========================
# VALIDASI
# =========================
if any(c in a_in_context for c in [ContextType.TIMING, ContextType.KIAI, ContextType.MAP, ContextType.SV, ContextType.GD, ContextType.NO_HS]) or add_to_beatmap:
    assert os.path.exists(input_beatmap), "Please upload a reference beatmap."

assert os.path.exists(input_audio), "Please upload an audio file."

if a_config == "v30":
    assert a_gamemode == 0, "V30 only supports standard mode."
    if any(c in a_in_context for c in [ContextType.KIAI, ContextType.MAP, ContextType.SV]):
        print("WARNING: V30 does not support KIAI, MAP, or SV in_context, ignoring.")
    if output_type != "[MAP]":
        print("WARNING: V30 only supports [MAP] output type, setting output type to [MAP].")
        a_output_type = [ContextType.MAP]
    if len(preset["descriptors"]) != 0 or len(preset["negative_descriptors"]) != 0:
        print("WARNING: V30 does not support descriptors or negative descriptors well.")
    if preset["super_timing"]:
        print("WARNING: V30 does not fully support super timing, generation will be VERY slow.")

# =========================
# SIAPKAN OUTPUT
# =========================
if os.path.exists(output_path):
    shutil.rmtree(output_path)
os.makedirs(output_path, exist_ok=True)

generated_osu_files = []

print(f"🚀 Generating {quantity} beatmap(s) using preset: {diff_preset}")
print("==================================================")

# =========================
# LOOP GENERATE
# =========================
for i in range(quantity):
    take_num = i + 1
    current_version = f"{a_base_version} (Take {take_num})" if quantity > 1 else a_base_version
    current_seed = random.randint(1, 999999) if a_seed is None else (a_seed + i)

    print(f"\n🎬 Take {take_num}/{quantity} | Diff: {current_version} | Seed: {current_seed}")

    try:
        try:
            from hydra.core.global_hydra import GlobalHydra
            GlobalHydra.instance().clear()
        except:
            pass

        with initialize_config_dir(version_base="1.1", config_dir="/content/Mapperatorinator/configs/inference"):
            conf = compose(config_name=a_config)

        # path
        conf.audio_path = input_audio
        conf.output_path = output_path
        conf.beatmap_path = input_beatmap

        # metadata
        conf.gamemode = a_gamemode
        conf.title = a_title
        conf.artist = a_artist
        conf.creator = a_creator
        conf.version = current_version
        conf.tags = a_tags
        conf.background = a_background
        conf.preview_time = a_preview_time

        # optional user overrides
        conf.mapper_id = a_mapper_id
        conf.beatmap_id = a_beatmap_id
        conf.year = a_year
        conf.lora_path = a_lora_path
        conf.hitsounded = hitsounded
        conf.generate_positions = generate_positions

        # preset difficulty block
        conf.difficulty = preset["difficulty"]
        conf.hp_drain_rate = preset["hp_drain_rate"]
        conf.circle_size = preset["circle_size"]
        conf.overall_difficulty = preset["overall_difficulty"]
        conf.approach_rate = preset["approach_rate"]
        conf.slider_multiplier = preset["slider_multiplier"]
        conf.slider_tick_rate = preset["slider_tick_rate"]

        # preset style
        conf.descriptors = preset["descriptors"]
        conf.negative_descriptors = preset["negative_descriptors"]

        # generation bounds/context
        conf.export_osz = False
        conf.add_to_beatmap = add_to_beatmap
        conf.start_time = a_start_time
        conf.end_time = a_end_time
        conf.in_context = a_in_context
        conf.output_type = a_output_type
        conf.seed = current_seed

        # preset sampling
        conf.cfg_scale = preset["cfg_scale"]
        conf.temperature = preset["temperature"]
        conf.top_p = preset["top_p"]
        conf.top_k = preset["top_k"]
        conf.num_beams = preset["num_beams"]
        conf.do_sample = preset["do_sample"]
        conf.timeshift_bias = preset["timeshift_bias"]

        # preset timing
        conf.timing_leniency = preset["timing_leniency"]
        conf.timing_temperature = preset["timing_temperature"]
        conf.timer_cfg_scale = preset["timer_cfg_scale"]
        conf.timer_num_beams = preset["timer_num_beams"]
        conf.timer_iterations = preset["timer_iterations"]
        conf.timer_bpm_threshold = preset["timer_bpm_threshold"]
        conf.super_timing = preset["super_timing"]
        conf.bpm = preset["bpm"]
        conf.offset = preset["offset"]
        conf.resnap_events = preset["resnap_events"]

        # preset positions
        conf.diff_cfg_scale = preset["diff_cfg_scale"]
        conf.refine_iters = preset["refine_iters"]
        conf.random_init = preset["random_init"]

        # long-map consistency
        conf.lookback = preset["lookback"]
        conf.lookahead = preset["lookahead"]

        _, result_path, _ = main(conf)

        if result_path and result_path.endswith(".osu"):
            generated_osu_files.append(result_path)
            print(f"✅ Finished Take {take_num}")

    except Exception as e:
        print(f"❌ Error on Take {take_num}: {e}")

# =========================
# PACKAGING
# =========================
if generated_osu_files:
    if download_mode == "Master .osz (Lagu + Semua Map)":
        file_title = a_title if a_title else "Unknown Title"
        file_artist = a_artist if a_artist else "Unknown Artist"
        master_osz_name = f"{file_artist} - {file_title} ({diff_preset} Batch).osz"

        with zipfile.ZipFile(master_osz_name, "w", zipfile.ZIP_DEFLATED) as zipf:
            zipf.write(input_audio, os.path.basename(input_audio))
            if a_background and os.path.exists(a_background):
                zipf.write(a_background, os.path.basename(a_background))
            for osu_file in generated_osu_files:
                zipf.write(osu_file, os.path.basename(osu_file))

        print(f"🎉 Downloading: {master_osz_name}")
        files.download(master_osz_name)

    else:
        for osu_file in generated_osu_files:
            print(f"Downloading: {os.path.basename(osu_file)}")
            files.download(osu_file)
            time.sleep(1.5)
else:
    print("❌ No .osu file generated.")